In [ ]:
# Cell 1: Imports and Configuration
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import shutil
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
ORIGINAL_DATASET = r"F:\SIH2025\dataset\Indian_bovine_breeds"
SUBSET_DATASET = r"F:\SIH2025\dataset\training_subset"

# Selected breeds from your Step 1 results
SELECTED_BREEDS = ['Sahiwal', 'Gir', 'Holstein_Friesian', 'Ayrshire', 'Brown_Swiss']
IMAGES_PER_BREED = 50  # Increase for better training (250 total images)

print("🚀 CATTLE BREED RECOGNITION - MODEL TRAINING PIPELINE")
print("="*70)
print(f"Selected breeds: {', '.join(SELECTED_BREEDS)}")
print(f"Images per breed: {IMAGES_PER_BREED}")
print(f"Total training images: {len(SELECTED_BREEDS) * IMAGES_PER_BREED}")

# Verify paths
if os.path.exists(ORIGINAL_DATASET):
    print("✅ Original dataset found")
else:
    print("❌ Original dataset not found - check path!")

print("✅ Setup complete - ready for next step!")

🚀 CATTLE BREED RECOGNITION - MODEL TRAINING PIPELINE
Selected breeds: Sahiwal, Gir, Holstein_Friesian, Ayrshire, Brown_Swiss
Images per breed: 50
Total training images: 250
✅ Original dataset found
✅ Setup complete - ready for next step!


In [ ]:
# Cell 2: Create Training Subset
def create_training_subset():
    """Create balanced training dataset"""
    print(f"\n📁 Creating training subset at: {SUBSET_DATASET}")
    
    # Remove existing subset if it exists
    if os.path.exists(SUBSET_DATASET):
        print("Removing existing subset...")
        shutil.rmtree(SUBSET_DATASET)
    
    os.makedirs(SUBSET_DATASET, exist_ok=True)
    
    summary = {}
    total_copied = 0
    
    for breed in SELECTED_BREEDS:
        print(f"\n🔄 Processing {breed}...")
        
        # Create breed folder
        breed_subset_path = os.path.join(SUBSET_DATASET, breed)
        os.makedirs(breed_subset_path, exist_ok=True)
        
        # Source folder
        source_path = os.path.join(ORIGINAL_DATASET, breed)
        
        if not os.path.exists(source_path):
            print(f"❌ {breed} folder not found at {source_path}")
            continue
        
        # Get all images
        all_images = [f for f in os.listdir(source_path) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        print(f"   Found {len(all_images)} images")
        
        # Select random subset
        random.seed(42)  # For reproducible results
        selected = random.sample(all_images, min(IMAGES_PER_BREED, len(all_images)))
        
        # Copy images
        copied = 0
        for img in selected:
            try:
                src = os.path.join(source_path, img)
                dst = os.path.join(breed_subset_path, img)
                shutil.copy2(src, dst)
                copied += 1
            except Exception as e:
                print(f"   Error copying {img}: {e}")
        
        summary[breed] = copied
        total_copied += copied
        print(f"   ✅ {breed}: {copied}/{len(selected)} images copied")
    
    print(f"\n🎉 Training subset created!")
    print(f"   Location: {SUBSET_DATASET}")
    print(f"   Total images: {total_copied}")
    print(f"   Breakdown: {summary}")
    
    return total_copied > 0

# Create the subset
success = create_training_subset()

if success:
    print("✅ Subset creation successful - ready for data loading!")
else:
    print("❌ Subset creation failed - check paths and permissions")


📁 Creating training subset at: F:\SIH2025\dataset\training_subset
Removing existing subset...

🔄 Processing Sahiwal...
   Found 439 images
   ✅ Sahiwal: 50/50 images copied

🔄 Processing Gir...
   Found 372 images
   ✅ Gir: 50/50 images copied

🔄 Processing Holstein_Friesian...
   Found 328 images
   ✅ Holstein_Friesian: 50/50 images copied

🔄 Processing Ayrshire...
   Found 234 images
   ✅ Ayrshire: 50/50 images copied

🔄 Processing Brown_Swiss...
   Found 225 images
   ✅ Brown_Swiss: 50/50 images copied

🎉 Training subset created!
   Location: F:\SIH2025\dataset\training_subset
   Total images: 250
   Breakdown: {'Sahiwal': 50, 'Gir': 50, 'Holstein_Friesian': 50, 'Ayrshire': 50, 'Brown_Swiss': 50}
✅ Subset creation successful - ready for data loading!


In [ ]:
# Cell 3: Load and Split Data
def load_and_split_data():
    """Load images and split into train/val/test"""
    print(f"\n📂 Loading data from: {SUBSET_DATASET}")
    
    # Try different image loading methods
    try:
        import cv2
        USE_CV2 = True
        print("✅ Using OpenCV for image loading")
    except ImportError:
        USE_CV2 = False
        print("⚠️ Using PIL for image loading")
        from PIL import Image
    
    # Load images
    images = []
    labels = []
    
    for breed_idx, breed in enumerate(SELECTED_BREEDS):
        breed_path = os.path.join(SUBSET_DATASET, breed)
        
        if not os.path.exists(breed_path):
            print(f"⚠️ {breed} folder not found")
            continue
        
        image_files = [f for f in os.listdir(breed_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        print(f"📸 Loading {breed}: {len(image_files)} images")
        
        loaded_count = 0
        for img_file in image_files:
            img_path = os.path.join(breed_path, img_file)
            
            try:
                if USE_CV2:
                    img = cv2.imread(img_path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img = cv2.resize(img, (224, 224))
                else:
                    from PIL import Image
                    with Image.open(img_path) as pil_img:
                        img = pil_img.convert('RGB')
                        img = img.resize((224, 224))
                        img = np.array(img)
                
                # Normalize to 0-1
                img = img.astype(np.float32) / 255.0
                
                images.append(img)
                labels.append(breed_idx)
                loaded_count += 1
                
                # Progress update
                if loaded_count % 10 == 0:
                    print(f"   Loaded {loaded_count} images...")
                    
            except Exception as e:
                print(f"   Error loading {img_file}: {e}")
        
        print(f"   ✅ {breed}: {loaded_count} images loaded successfully")
    
    if len(images) == 0:
        print("❌ No images loaded!")
        return None
    
    X = np.array(images)
    y = np.array(labels)
    
    print(f"\n📊 Dataset Summary:")
    print(f"   Total images: {X.shape[0]}")
    print(f"   Image shape: {X.shape[1:]}")
    print(f"   Number of breeds: {len(SELECTED_BREEDS)}")
    print(f"   Images per breed: {np.bincount(y)}")
    
    # Split data
    from sklearn.model_selection import train_test_split
    
    print(f"\n✂️ Splitting data...")
    
    # First split: separate test set (20%)
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Second split: separate train (70%) and validation (10%)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp  # 0.125 of 80% = 10% of total
    )
    
    print(f"✅ Data split complete:")
    print(f"   Training: {X_train.shape[0]} images ({X_train.shape[0]/len(X)*100:.0f}%)")
    print(f"   Validation: {X_val.shape[0]} images ({X_val.shape[0]/len(X)*100:.0f}%)")
    print(f"   Test: {X_test.shape[0]} images ({X_test.shape[0]/len(X)*100:.0f}%)")
    
    # Store data globally for next cells
    globals()['X_train'] = X_train
    globals()['X_val'] = X_val  
    globals()['X_test'] = X_test
    globals()['y_train'] = y_train
    globals()['y_val'] = y_val
    globals()['y_test'] = y_test
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Load and split the data
data_splits = load_and_split_data()

if data_splits is not None:
    print("✅ Data loading and splitting successful!")
    print("✅ Ready for model training!")
else:
    print("❌ Data loading failed - check previous steps")


📂 Loading data from: F:\SIH2025\dataset\training_subset
✅ Using OpenCV for image loading
📸 Loading Sahiwal: 50 images
   Loaded 10 images...
   Loaded 20 images...
   Loaded 30 images...
   Loaded 40 images...
   Loaded 50 images...
   ✅ Sahiwal: 50 images loaded successfully
📸 Loading Gir: 50 images
   Loaded 10 images...
   Loaded 20 images...
   Loaded 30 images...
   Loaded 40 images...
   Loaded 50 images...
   ✅ Gir: 50 images loaded successfully
📸 Loading Holstein_Friesian: 50 images
   Loaded 10 images...
   Loaded 20 images...
   Loaded 30 images...
   Loaded 40 images...
   Loaded 50 images...
   ✅ Holstein_Friesian: 50 images loaded successfully
📸 Loading Ayrshire: 50 images
   Loaded 10 images...
   Loaded 20 images...
   Loaded 30 images...
   Loaded 40 images...
   Loaded 50 images...
   ✅ Ayrshire: 50 images loaded successfully
📸 Loading Brown_Swiss: 50 images
   Loaded 10 images...
   Loaded 20 images...
   Loaded 30 images...
   Loaded 40 images...
   Loaded 50 images

In [ ]:
# Cell 4: Traditional ML Training
def train_traditional_ml():
    """Train traditional ML models with different features"""
    print(f"\n🤖 TRADITIONAL MACHINE LEARNING")
    print("="*50)
    
    # Check if data is available
    if 'X_train' not in globals():
        print("❌ Data not loaded. Run previous cells first!")
        return None
    
    # Import ML libraries
    try:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.svm import SVC
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.metrics import accuracy_score, classification_report
        from sklearn.model_selection import cross_val_score
    except ImportError:
        print("📦 Installing scikit-learn...")
        os.system("pip install scikit-learn")
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.svm import SVC
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.metrics import accuracy_score, classification_report
        from sklearn.model_selection import cross_val_score
    
    # Feature extraction functions
    def extract_hog_features(images):
        """Extract HOG features"""
        try:
            from skimage.feature import hog
        except ImportError:
            print("📦 Installing scikit-image...")
            os.system("pip install scikit-image")
            from skimage.feature import hog
        
        print("🔍 Extracting HOG features...")
        features = []
        
        for i, img in enumerate(images):
            if (i+1) % 20 == 0:
                print(f"   Processed {i+1}/{len(images)} images")
            
            # Convert to grayscale
            gray = np.dot(img[...,:3], [0.2989, 0.5870, 0.1140])
            
            try:
                # Extract HOG features
                feat = hog(gray, pixels_per_cell=(16, 16), cells_per_block=(2, 2), 
                          visualize=False, feature_vector=True)
                features.append(feat)
            except Exception as e:
                print(f"   Error processing image {i}: {e}")
                if len(features) > 0:
                    features.append(np.zeros_like(features[0]))
                else:
                    features.append(np.zeros(1764))  # Default size
        
        return np.array(features)
    
    def extract_color_features(images):
        """Extract color histogram features"""
        print("🎨 Extracting color histogram features...")
        features = []
        
        for i, img in enumerate(images):
            if (i+1) % 20 == 0:
                print(f"   Processed {i+1}/{len(images)} images")
            
            # RGB histograms
            hist_r = np.histogram(img[:,:,0], bins=32, range=(0,1))[0]
            hist_g = np.histogram(img[:,:,1], bins=32, range=(0,1))[0] 
            hist_b = np.histogram(img[:,:,2], bins=32, range=(0,1))[0]
            
            feat = np.concatenate([hist_r, hist_g, hist_b])
            features.append(feat)
        
        return np.array(features)
    
    # Models to test
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'SVM (RBF)': SVC(kernel='rbf', random_state=42, probability=True),
        'KNN (k=5)': KNeighborsClassifier(n_neighbors=5)
    }
    
    # Feature methods
    feature_methods = {
        'HOG': extract_hog_features,
        'Color Histograms': extract_color_features,
        'Raw Pixels': lambda x: x.reshape(len(x), -1)
    }
    
    results = []
    best_model_info = None
    best_accuracy = 0
    
    for feat_name, feat_extractor in feature_methods.items():
        print(f"\n--- Testing with {feat_name} ---")
        
        try:
            # Extract features
            X_train_feat = feat_extractor(X_train)
            X_val_feat = feat_extractor(X_val)
            
            print(f"Feature shape: {X_train_feat.shape}")
            
            # Test each model
            for model_name, model in models.items():
                try:
                    print(f"  Training {model_name}...")
                    
                    # Train model
                    model.fit(X_train_feat, y_train)
                    
                    # Validate
                    y_pred = model.predict(X_val_feat)
                    accuracy = accuracy_score(y_val, y_pred)
                    
                    # Cross-validation
                    cv_scores = cross_val_score(model, X_train_feat, y_train, cv=3)
                    
                    result = {
                        'Model': model_name,
                        'Features': feat_name,
                        'Accuracy': accuracy,
                        'CV_Mean': cv_scores.mean(),
                        'CV_Std': cv_scores.std()
                    }
                    results.append(result)
                    
                    print(f"    ✅ Accuracy: {accuracy:.3f}, CV: {cv_scores.mean():.3f}±{cv_scores.std():.3f}")
                    
                    # Track best model
                    if accuracy > best_accuracy:
                        best_accuracy = accuracy
                        best_model_info = (model_name, feat_name, model, X_train_feat, X_val_feat)
                    
                except Exception as e:
                    print(f"    ❌ Error with {model_name}: {e}")
        
        except Exception as e:
            print(f"  ❌ Error with {feat_name} features: {e}")
    
    # Results summary
    if results:
        df_results = pd.DataFrame(results)
        df_results = df_results.sort_values('Accuracy', ascending=False)
        
        print(f"\n📊 TRADITIONAL ML RESULTS:")
        print("="*60)
        print(df_results.to_string(index=False))
        
        # Best model
        if best_model_info:
            model_name, feat_name, model, X_train_feat, X_val_feat = best_model_info
            print(f"\n🏆 BEST TRADITIONAL ML:")
            print(f"   {model_name} + {feat_name}")
            print(f"   Validation Accuracy: {best_accuracy:.3f} ({best_accuracy*100:.1f}%)")
            
            # Test on test set
            try:
                if feat_name == 'HOG':
                    X_test_feat = extract_hog_features(X_test)
                elif feat_name == 'Color Histograms':
                    X_test_feat = extract_color_features(X_test)
                else:
                    X_test_feat = X_test.reshape(len(X_test), -1)
                
                y_test_pred = model.predict(X_test_feat)
                test_accuracy = accuracy_score(y_test, y_test_pred)
                
                print(f"   Test Accuracy: {test_accuracy:.3f} ({test_accuracy*100:.1f}%)")
                print(f"\n📋 Detailed Test Results:")
                print(classification_report(y_test, y_test_pred, target_names=SELECTED_BREEDS))
                
                # Store best traditional ML results
                globals()['best_traditional_ml'] = {
                    'name': f"{model_name} + {feat_name}",
                    'accuracy': test_accuracy,
                    'model': model
                }
                
            except Exception as e:
                print(f"   ❌ Error testing on test set: {e}")
    
    else:
        print("❌ No successful results from traditional ML")
    
    return df_results if results else None

# Train traditional ML models
ml_results = train_traditional_ml()

if ml_results is not None:
    print("✅ Traditional ML training complete!")
    print("✅ Ready for deep learning models!")
else:
    print("❌ Traditional ML training failed")


🤖 TRADITIONAL MACHINE LEARNING

--- Testing with HOG ---
🔍 Extracting HOG features...
   Processed 20/175 images
   Processed 40/175 images
   Processed 60/175 images
   Processed 80/175 images
   Processed 100/175 images
   Processed 120/175 images
   Processed 140/175 images
   Processed 160/175 images
🔍 Extracting HOG features...
   Processed 20/25 images
Feature shape: (175, 6084)
  Training Random Forest...
    ✅ Accuracy: 0.240, CV: 0.354±0.026
  Training SVM (RBF)...
    ✅ Accuracy: 0.440, CV: 0.315±0.031
  Training KNN (k=5)...
    ✅ Accuracy: 0.280, CV: 0.229±0.036

--- Testing with Color Histograms ---
🎨 Extracting color histogram features...
   Processed 20/175 images
   Processed 40/175 images
   Processed 60/175 images
   Processed 80/175 images
   Processed 100/175 images
   Processed 120/175 images
   Processed 140/175 images
   Processed 160/175 images
🎨 Extracting color histogram features...
   Processed 20/25 images
Feature shape: (175, 96)
  Training Random Forest..

In [ ]:
# Cell 5: CNN Training
def train_cnn_models():
    """Train CNN models"""
    print(f"\n🧠 DEEP LEARNING (CNN)")
    print("="*50)
    
    # Check if data is available
    if 'X_train' not in globals():
        print("❌ Data not loaded. Run previous cells first!")
        return None
    
    # Import TensorFlow
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
        from tensorflow.keras.preprocessing.image import ImageDataGenerator
        print(f"✅ TensorFlow {tf.__version__} available")
    except ImportError:
        print("📦 Installing TensorFlow...")
        os.system("pip install tensorflow")
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
        from tensorflow.keras.preprocessing.image import ImageDataGenerator
    
    print(f"Using TensorFlow {tf.__version__}")
    
    # Model 1: Simple CNN
    def create_simple_cnn():
        """Create a simple CNN model"""
        model = Sequential([
            Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
            MaxPooling2D(2, 2),
            
            Conv2D(64, (3, 3), activation='relu'),
            MaxPooling2D(2, 2),
            
            Conv2D(128, (3, 3), activation='relu'),
            MaxPooling2D(2, 2),
            
            Flatten(),
            Dense(512, activation='relu'),
            Dropout(0.5),
            Dense(len(SELECTED_BREEDS), activation='softmax')
        ])
        
        model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])
        
        return model
    
    # Model 2: Transfer Learning - MobileNetV2
    def create_mobilenet_model():
        """Create MobileNetV2 transfer learning model"""
        try:
            from tensorflow.keras.applications import MobileNetV2
            from tensorflow.keras.layers import GlobalAveragePooling2D
            from tensorflow.keras.models import Model
            
            # Base model
            base_model = MobileNetV2(weights='imagenet', include_top=False, 
                                   input_shape=(224, 224, 3))
            base_model.trainable = False  # Freeze base model
            
            # Add custom head
            x = base_model.output
            x = GlobalAveragePooling2D()(x)
            x = Dense(256, activation='relu')(x)
            x = Dropout(0.5)(x)
            predictions = Dense(len(SELECTED_BREEDS), activation='softmax')(x)
            
            model = Model(inputs=base_model.input, outputs=predictions)
            model.compile(optimizer='adam',
                         loss='sparse_categorical_crossentropy', 
                         metrics=['accuracy'])
            
            return model
        except Exception as e:
            print(f"❌ Error creating MobileNetV2 model: {e}")
            return None
    
    # Data augmentation
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
        fill_mode='nearest'
    )
    
    cnn_results = []
    
    # Train Simple CNN
    print(f"\n🔥 Training Simple CNN...")
    try:
        simple_cnn = create_simple_cnn()
        print(f"Model parameters: {simple_cnn.count_params():,}")
        
        # Train
        print("Training for 5 epochs...")
        history_simple = simple_cnn.fit(
            datagen.flow(X_train, y_train, batch_size=16),
            epochs=5,
            validation_data=(X_val, y_val),
            verbose=1
        )
        
        # Evaluate
        simple_test_loss, simple_test_acc = simple_cnn.evaluate(X_test, y_test, verbose=0)
        print(f"✅ Simple CNN Test Accuracy: {simple_test_acc:.3f} ({simple_test_acc*100:.1f}%)")
        
        cnn_results.append({
            'Model': 'Simple CNN',
            'Parameters': simple_cnn.count_params(),
            'Test_Accuracy': simple_test_acc,
            'Test_Loss': simple_test_loss
        })
        
        # Store model
        globals()['simple_cnn_model'] = simple_cnn
        
    except Exception as e:
        print(f"❌ Simple CNN training failed: {e}")
    
    # Train MobileNetV2
    print(f"\n📱 Training MobileNetV2 (Transfer Learning)...")
    try:
        mobilenet_model = create_mobilenet_model()
        
        if mobilenet_model is not None:
            print(f"Model parameters: {mobilenet_model.count_params():,}")
            
            # Train
            print("Training for 5 epochs...")
            history_mobile = mobilenet_model.fit(
                datagen.flow(X_train, y_train, batch_size=16),
                epochs=5,
                validation_data=(X_val, y_val),
                verbose=1
            )
            
            # Evaluate
            mobile_test_loss, mobile_test_acc = mobilenet_model.evaluate(X_test, y_test, verbose=0)
            print(f"✅ MobileNetV2 Test Accuracy: {mobile_test_acc:.3f} ({mobile_test_acc*100:.1f}%)")
            
            cnn_results.append({
                'Model': 'MobileNetV2 (Transfer)',
                'Parameters': mobilenet_model.count_params(),
                'Test_Accuracy': mobile_test_acc,
                'Test_Loss': mobile_test_loss
            })
            
            # Store model
            globals()['mobilenet_model'] = mobilenet_model
            
    except Exception as e:
        print(f"❌ MobileNetV2 training failed: {e}")
    
    # Results summary
    if cnn_results:
        df_cnn = pd.DataFrame(cnn_results)
        df_cnn = df_cnn.sort_values('Test_Accuracy', ascending=False)
        
        print(f"\n📊 CNN RESULTS:")
        print("="*60)
        print(df_cnn.to_string(index=False))
        
        # Best CNN
        best_cnn = df_cnn.iloc[0]
        print(f"\n🏆 BEST CNN MODEL:")
        print(f"   {best_cnn['Model']}")
        print(f"   Test Accuracy: {best_cnn['Test_Accuracy']:.3f} ({best_cnn['Test_Accuracy']*100:.1f}%)")
        print(f"   Parameters: {best_cnn['Parameters']:,}")
        
        # Store best CNN results
        globals()['best_cnn'] = {
            'name': best_cnn['Model'],
            'accuracy': best_cnn['Test_Accuracy'],
            'parameters': best_cnn['Parameters']
        }
        
        return df_cnn
    
    else:
        print("❌ No successful CNN results")
        return None

# Train CNN models
cnn_results = train_cnn_models()

if cnn_results is not None:
    print("✅ CNN training complete!")
    print("✅ Ready for final comparison!")
else:
    print("❌ CNN training failed")


🧠 DEEP LEARNING (CNN)
✅ TensorFlow 2.20.0 available
Using TensorFlow 2.20.0

🔥 Training Simple CNN...
Model parameters: 44,398,661
Training for 5 epochs...
Epoch 1/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.2286 - loss: 2.3106 - val_accuracy: 0.3200 - val_loss: 1.5991
Epoch 2/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 11s 943ms/step - accuracy: 0.2457 - loss: 1.5637 - val_accuracy: 0.3600 - val_loss: 1.5335
Epoch 3/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 875ms/step - accuracy: 0.2629 - loss: 1.6291 - val_accuracy: 0.2400 - val_loss: 1.6456
Epoch 4/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 916ms/step - accuracy: 0.2743 - loss: 1.6235 - val_accuracy: 0.3600 - val_loss: 1.5825
Epoch 5/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.2343 - loss: 1.5736 - val_accuracy: 0.2800 - val_loss: 1.5613
✅ Simple CNN Test Accuracy: 0.300 (30.0%)

📱 Training MobileNetV2 (Transfer Learning)...
Model parameters: 2,587,205
Training for 5 epochs...
Epoch 1/5
11/11 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.2343

In [13]:
# Cell 6: Final Results Comparison and Recommendations
def final_comparison_and_recommendations():
    """Compare all models and provide recommendations"""
    print(f"\n🏆 FINAL RESULTS COMPARISON")
    print("="*70)
    
    all_results = []
    
    # Get Traditional ML results
    if 'best_traditional_ml' in globals():
        ml_info = globals()['best_traditional_ml']
        all_results.append({
            'Category': 'Traditional ML',
            'Model': ml_info['name'],
            'Accuracy': ml_info['accuracy'],
            'Parameters': 'N/A',
            'Mobile_Friendly': '⭐⭐⭐⭐⭐'
        })
        print(f"✅ Traditional ML: {ml_info['name']} - {ml_info['accuracy']:.3f}")
    else:
        print("⚠️ No Traditional ML results available")
    
    # Get CNN results
    if 'best_cnn' in globals():
        cnn_info = globals()['best_cnn']
        mobile_friendly = '⭐⭐⭐⭐⭐' if 'MobileNet' in cnn_info['name'] else '⭐⭐⭐'
        
        all_results.append({
            'Category': 'Deep Learning',
            'Model': cnn_info['name'],
            'Accuracy': cnn_info['accuracy'],
            'Parameters': f"{cnn_info['parameters']:,}",
            'Mobile_Friendly': mobile_friendly
        })
        print(f"✅ Deep Learning: {cnn_info['name']} - {cnn_info['accuracy']:.3f}")
    else:
        print("⚠️ No CNN results available")
    
    if not all_results:
        print("❌ No results to compare! Run previous cells first.")
        return
    
    # Create comparison DataFrame
    df_comparison = pd.DataFrame(all_results)
    df_comparison = df_comparison.sort_values('Accuracy', ascending=False)
    
    print(f"\n📊 COMPLETE COMPARISON TABLE:")
    print("="*80)
    print(df_comparison.to_string(index=False))
    
    # Best overall model
    best_overall = df_comparison.iloc[0]
    
    print(f"\n🏆 WINNER:")
    print(f"   Model: {best_overall['Model']}")
    print(f"   Accuracy: {best_overall['Accuracy']:.3f} ({best_overall['Accuracy']*100:.1f}%)")
    print(f"   Category: {best_overall['Category']}")
    print(f"   Mobile Friendly: {best_overall['Mobile_Friendly']}")
    
    # Performance interpretation
    accuracy = best_overall['Accuracy']
    if accuracy >= 0.85:
        performance = "🎉 EXCELLENT!"
        description = "Outstanding performance for a 5-breed classifier!"
    elif accuracy >= 0.75:
        performance = "👍 VERY GOOD!"
        description = "Solid performance, ready for deployment."
    elif accuracy >= 0.65:
        performance = "✅ GOOD!"
        description = "Acceptable performance, room for improvement."
    else:
        performance = "🔧 NEEDS WORK"
        description = "Consider more data or different approaches."
    
    print(f"\n📈 PERFORMANCE ASSESSMENT:")
    print(f"   {performance}")

In [14]:
final_comparison_and_recommendations()


🏆 FINAL RESULTS COMPARISON
✅ Traditional ML: Random Forest + Raw Pixels - 0.340
✅ Deep Learning: MobileNetV2 (Transfer) - 0.580

📊 COMPLETE COMPARISON TABLE:
      Category                      Model  Accuracy Parameters Mobile_Friendly
 Deep Learning     MobileNetV2 (Transfer)      0.58  2,587,205           ⭐⭐⭐⭐⭐
Traditional ML Random Forest + Raw Pixels      0.34        N/A           ⭐⭐⭐⭐⭐

🏆 WINNER:
   Model: MobileNetV2 (Transfer)
   Accuracy: 0.580 (58.0%)
   Category: Deep Learning
   Mobile Friendly: ⭐⭐⭐⭐⭐

📈 PERFORMANCE ASSESSMENT:
   🔧 NEEDS WORK
